In [40]:
import requests
import pandas as pd
import json
from dateutil import parser

In [7]:
API_KEY = "ff8b7c45e33547101220ed64c14e79db-6e2019047f6bd7293fa46fed157a4a64"
ACCOUNT_ID = "101-001-30914963-001"
OANDA_URL = "https://api-fxpractice.oanda.com/v3"


In [8]:
session = requests.Session()

In [9]:
session.headers.update({
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"})

In [10]:
params = {
    "count": 10,
    "granularity": "H1",
    "price": "MBA",
}

In [11]:
url = f"{OANDA_URL}/accounts/{ACCOUNT_ID}/instruments"

In [12]:
response = session.get(url, params=params, data=None , headers=None)

In [13]:
response.status_code

200

In [15]:
data = response.json()

In [17]:
instruments_list = data["instruments"]

In [19]:
len(instruments_list)

68

In [21]:
instruments_list[0].keys()

dict_keys(['name', 'type', 'displayName', 'pipLocation', 'displayPrecision', 'tradeUnitsPrecision', 'minimumTradeSize', 'maximumTrailingStopDistance', 'minimumTrailingStopDistance', 'maximumPositionSize', 'maximumOrderUnits', 'marginRate', 'guaranteedStopLossOrderMode', 'tags', 'financing'])

In [22]:
key_i=['name', 'type', 'displayName', 'pipLocation', 'displayPrecision', 'tradeUnitsPrecision', 
       'marginRate']

In [23]:
instruments_dict = {}
for i in instruments_list:
    key = i["name"]
    instruments_dict[key] = {k: i[k] for k in key_i}

In [25]:
instruments_dict['USD_CAD']

{'name': 'USD_CAD',
 'type': 'CURRENCY',
 'displayName': 'USD/CAD',
 'pipLocation': -4,
 'displayPrecision': 5,
 'tradeUnitsPrecision': 0,
 'marginRate': '0.02'}

In [31]:
with open('../data/instruments.json', 'w') as f:
    json.dump(instruments_dict, f, indent=2)


In [32]:
def fetch_candles(pair_name,count = 10, granularity ="H1"):
    url = f"{OANDA_URL}/instruments/{pair_name}/candles"
    params = {
        "count": count,
        "granularity": granularity,
        "price": "MBA",
    }
    response = session.get(url, params=params, data=None , headers=None)
    data = response.json()
    if response.status_code == 200:
        if 'candles' not in data:
            data = []
        else:
            data = data['candles']
    return response.status_code, data

In [33]:
code, data = fetch_candles('EUR_USD', count=10, granularity='H1')

In [36]:
len(data)

10

In [46]:
final_data = []
for candle in data:
    if candle['complete']== False:
        continue
    new_dict = {}
    new_dict['time'] = parser.parse(candle['time'])
    new_dict['volume'] = candle['volume']
    final_data.append(new_dict)
df = pd.DataFrame.from_dict(final_data)

In [47]:
len(df)

10

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype                  
---  ------  --------------  -----                  
 0   time    10 non-null     datetime64[ns, tzutc()]
 1   volume  10 non-null     int64                  
dtypes: datetime64[ns, tzutc()](1), int64(1)
memory usage: 292.0 bytes


In [50]:
df

,time,volume
0,2025-01-24 12:00:00+00:00,6462
1,2025-01-24 13:00:00+00:00,8141
2,2025-01-24 14:00:00+00:00,12331
3,2025-01-24 15:00:00+00:00,16377
4,2025-01-24 16:00:00+00:00,10021
5,2025-01-24 17:00:00+00:00,6300
6,2025-01-24 18:00:00+00:00,4373
7,2025-01-24 19:00:00+00:00,3629
8,2025-01-24 20:00:00+00:00,3024
9,2025-01-24 21:00:00+00:00,2121
